In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from dataclasses import dataclass
import numpy as np
from typing import Any, List, Callable
from collections import defaultdict
from mdp import Policy, MDP, Step, Rollout, FlackyTramMDP, RLAlgorithm, walk_tram_exploration_policy, simulate
from functools import partial
import torch 
import torch.nn as nn

def one_hot(index: int, num_classes: int) -> torch.Tensor:
    v = torch.zeros(num_classes)
    v[index] = 1.0
    return v

print(one_hot(1, 10))
np.random.seed(42)
torch.random.manual_seed(42)
tram_mdp = FlackyTramMDP(num_locs=10, failure_prob=0.4)

tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0.])


In [ ]:
class ParameterizedQLearning(RLAlgorithm):
    def __init__(
            self, 
            num_locs: int,
            actions: List[str],
            exploration_policy: Policy, 
            epsilon: float = 0.4, 
            lr: float = 0.1, 
            discount: float = 1.0):
        self.exploration_policy = exploration_policy
        self.epsilon = epsilon
        self.lr = lr
        self.discount = discount
        self.num_features = num_locs * len(actions)
        self.actions = actions
        self.model = nn.Linear(self.num_features, 1)
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr)

    def phi(self, state: int, action: str) -> torch.Tensor:
        index = (state - 1) * len(self.actions) + self.actions.index(action)
        return one_hot(index, self.num_features)

    def Q(self, state: int, action: str) -> torch.Tensor:
        return self.model(self.phi(state, action))

    def pi(self, state: int) -> str:
        q_values = { action : self.Q(state, action) for action in self.actions}
        return max(q_values.keys(), key=lambda s: q_values[s].item())

    def get_action(self, state: int) -> str:
        if torch.rand(1).item() < self.epsilon:
            return self.exploration_policy(state)
        else:
            return self.pi(state)
        
    def incorporate_feedback(self, state: int, action: str, reward: float, next_state: int, is_end: bool):
        if is_end:
            target = reward
        else:
            next_action = self.pi(next_state)
            target = reward + self.discount * self.Q(next_state, next_action)

        value = self.Q(state, action)
        # use squared loss: Loss = (Q(s,a) - target)^2
        loss = (value - target) ** 2
        # Update model with a gradient step
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

# Test ParameterizedQLearning
exploration_policy = partial(walk_tram_exploration_policy, tram_mdp.num_locs)
rl = ParameterizedQLearning(tram_mdp.num_locs, ["walk", "tram"], exploration_policy, epsilon=0.4, lr=0.1, discount=0.9)
scores = simulate(tram_mdp, rl, num_trials=200)
print(f"[Parameterized Q-Learning] mean utility: {np.mean(scores):.2f}")
print("Policy Learned π(s):", {s: rl.pi(s) for s in range(1, tram_mdp.num_locs + 1)})

[Parameterized Q-Learning] mean utility: -6.63
Policy Learned π(s): {1: 'walk', 2: 'walk', 3: 'walk', 4: 'walk', 5: 'tram', 6: 'walk', 7: 'walk', 8: 'walk', 9: 'walk', 10: 'tram'}


In [ ]:
class Reinforce(RLAlgorithm):
    def __init__(self, num_locs: int, actions: List[str], lr: float = 0.1, discount: float = 1.0):
        self.lr = lr
        self.discount = discount
        self.num_locs = num_locs
        self.actions = actions
        self.model = nn.Linear(num_locs, len(actions))
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr)
        self.rollout = []
        self.utility = 0.0
        self.start_state = None

    def phi(self, state: int) -> torch.Tensor:
        return one_hot(state - 1, self.num_locs)

    def pi(self, state: int) -> dict[str, float]:
        # Return action-prob mapping: π_θ(a|s)
        logits = self.model(self.phi(state))
        probs = torch.softmax(logits, dim=0)
        return dict(zip(self.actions, probs.tolist()))

    def get_action(self, state: int) -> str:
        logits = self.model(self.phi(state))
        probs = torch.softmax(logits, dim=0)
        # sample an action with the softmax probablities 
        index = int(torch.multinomial(probs, num_samples=1).item())
        return self.actions[index]

    def incorporate_feedback(self, state: int, action: str, reward: float, next_state: int, is_end: bool):
        if self.start_state is None: 
            self.start_state = state
        self.utility += reward * self.discount ** len(self.rollout)
        self.rollout.append(Step(action, reward, 1.0, next_state))

        if is_end:
            loss = torch.tensor(0.0)
            for i, step in enumerate(self.rollout):
                s = self.start_state if i == 0 else self.rollout[i - 1].state
                # unsqueeze to add batch dimension. logits: (1, num_actions)
                logits = self.model(self.phi(s)).unsqueeze(0)
                # target: (1,)
                target = torch.tensor([self.actions.index(step.action)], dtype=torch.long)
                cross_entropy = nn.CrossEntropyLoss()
                #!!potential issue here - utility should NOT be negative! otherwise model will move away from correct action!
                loss += self.utility * cross_entropy(logits, target)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            #reset rollout
            self.utility = 0.0
            self.rollout = []
            self.start_state = None

# Test Reinforce
rl = Reinforce(tram_mdp.num_locs, ["walk", "tram"], discount=0.9, lr=0.01)
scores = simulate(tram_mdp, rl, num_trials=300)
print(f"[REINFORCE] mean utility: {np.mean(scores):.2f}")
print("Policy Learned π(a|s=1):", {i + 1: rl.pi(i + 1) for i in range(tram_mdp.num_locs)})

[REINFORCE] mean utility: -8.44
Policy Learned π(a|s=1): {1: {'walk': 0.9915908575057983, 'tram': 0.008409163914620876}, 2: {'walk': 0.9997114539146423, 'tram': 0.000288572337012738}, 3: {'walk': 0.9906818866729736, 'tram': 0.009318140335381031}, 4: {'walk': 0.9989872574806213, 'tram': 0.0010126930428668857}, 5: {'walk': 0.9987146854400635, 'tram': 0.001285374048165977}, 6: {'walk': 0.9999018907546997, 'tram': 9.809117182157934e-05}, 7: {'walk': 0.999931812286377, 'tram': 6.822454452048987e-05}, 8: {'walk': 0.9999468326568604, 'tram': 5.3134273912291974e-05}, 9: {'walk': 0.9998644590377808, 'tram': 0.0001354875712422654}, 10: {'walk': 0.9985470175743103, 'tram': 0.0014529264299198985}}
